# Same benchmarks on Google Colab (bonus: laptop vs Colab + roofline)

This notebook runs `src/machine_bench.py`, the project's portable benchmark suite, on a free Colab CPU
runtime. The laptop ran the exact same script and the same data (`results/machine_laptop.json`), so the two
JSON files can be compared directly with `src/machine_compare.py`.

**Steps**
1. *Runtime → Change runtime type → CPU* (no GPU needed; the suite never uses one).
2. On the laptop: `.venv/bin/python src/machine_bench.py --pack` creates `colab_pack.zip`
   (code + mpnet embeddings + Phase 1 reference, ~60 MB). Upload it to Google Drive as
   `MyDrive/hpp_colab/colab_pack.zip` (or set `USE_DRIVE = False` and pick the file when asked).
3. *Runtime → Run all*. The suite takes a few minutes (about 1 minute on the laptop).
4. The last cell downloads `machine_colab.json` (and copies it to the Drive folder).
   Put it into the project's `results/` folder and run `src/machine_compare.py`.

Same data, same code and the same benchmark rules (warmup, ≥ 5 runs, median) on both machines;
only the hardware and the BLAS library differ.

In [ ]:
# 1. Code + data: from Google Drive if colab_pack.zip is there, otherwise upload it from the computer
import pathlib, zipfile

USE_DRIVE = True
PROJECT = pathlib.Path("/content/project")
DRIVE_DIR = pathlib.Path("/content/drive/MyDrive/hpp_colab")

pack = None
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    if (DRIVE_DIR / "colab_pack.zip").exists():
        pack = DRIVE_DIR / "colab_pack.zip"
if pack is None:
    from google.colab import files
    print("Choose colab_pack.zip")
    pack = pathlib.Path(next(iter(files.upload())))

zipfile.ZipFile(pack).extractall(PROJECT)
print(pack, "->", PROJECT)
print(sorted(str(p.relative_to(PROJECT)) for p in PROJECT.rglob("*") if p.is_file()))

In [ ]:
# 2. What machine did we get?
!lscpu | grep -E "Model name|Socket|Core\(s\)|Thread\(s\)|L1d|L2|L3"
!nproc; free -h | head -2
import numpy, scipy, numba
print("numpy", numpy.__version__, "| scipy", scipy.__version__, "| numba", numba.__version__)
numpy.show_config()

In [ ]:
# 3. The benchmark suite (a few minutes). Progress is printed as it goes.
!cd /content/project && python src/machine_bench.py --name colab

In [ ]:
# 4. Keep the result: copy to Drive and download it
import shutil
out = PROJECT / "results" / "machine_colab.json"
if USE_DRIVE:
    DRIVE_DIR.mkdir(parents=True, exist_ok=True)
    shutil.copy(out, DRIVE_DIR / out.name)
    print("copied to", DRIVE_DIR / out.name)
from google.colab import files
files.download(str(out))